# 07 — Named Entity Recognition

**Learning objective.** Extract typed spans and understand precision/recall trade-offs with spaCy’s rule-based EntityRuler.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## 🧠 Visual engineering mental model

![Causal mindmap](assets/mindmaps/07_named_entity_recognition.svg)

Read the map **left → right**, then inspect the control knob above it. The learning goal is to predict how a control change propagates before running code.

## 🎛️ Change map — if you change this, what moves downstream?

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Change the **entity schema** | what counts as a target changes | training labels and evaluation change even on identical text |
| Require **exact boundaries** | evaluation becomes stricter | partial extraction no longer counts |
| Use rules vs learned NER | precision/coverage trade-offs change | stable formats favor rules; variable language favors learned models |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

## 🔮 Predict before you run

1. Is `New York University` one ORG span or a GPE plus word? Your schema decides.
2. If exact-match F1 drops but partial overlap is high, what kind of error is occurring?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Use NER when downstream systems need typed spans, not merely document labels.

### When not / caution
Do not create entity labels that have no operational use or cannot be annotated consistently.

### Debugging lens
Separate two failures: span boundary wrong vs type wrong—they require different fixes.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
import spacy
from spacy.pipeline import EntityRuler
nlp=spacy.blank('en')
ruler=nlp.add_pipe('entity_ruler')
ruler.add_patterns([
 {'label':'ORG','pattern':'OpenAI'},
 {'label':'ORG','pattern':'NielsenIQ'},
 {'label':'GPE','pattern':'Chennai'},
 {'label':'PRODUCT','pattern':[{'LOWER':'gpt'},{'IS_DIGIT':True}]},
])
text='OpenAI demonstrated GPT 5 in Chennai while NielsenIQ expanded its AI research.'
doc=nlp(text)
[(ent.text,ent.label_,ent.start_char,ent.end_char) for ent in doc.ents]

[('OpenAI', 'ORG', 0, 6),
 ('GPT 5', 'PRODUCT', 20, 25),
 ('Chennai', 'GPE', 29, 36),
 ('NielsenIQ', 'ORG', 43, 52)]

A production NER system must define an **entity schema** first. `ORG`, `PERSON`, `GPE`, `PRODUCT` are not universal truths—they are labels designed for a downstream task. Evaluation should use exact/partial span criteria deliberately.

In [3]:
expected={('OpenAI','ORG'),('Chennai','GPE'),('NielsenIQ','ORG')}
pred={(e.text,e.label_) for e in doc.ents}
print('expected subset recovered:', expected.issubset(pred))
print('predictions:', sorted(pred))

expected subset recovered: True
predictions: [('Chennai', 'GPE'), ('GPT 5', 'PRODUCT'), ('NielsenIQ', 'ORG'), ('OpenAI', 'ORG')]


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain NER as span detection + type classification
- Design entity schemas around downstream use